<a href="https://colab.research.google.com/github/pnperl/Heiken-Ashi-Screener-HA-reversal/blob/main/Copy_of_Heiken_Ashi_Screener_HA_reversal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance pandas numpy requests python-dotenv -q

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import time
import logging
from datetime import datetime, timedelta

from zoneinfo import ZoneInfo
from IPython.display import clear_output
from google.colab import userdata

# --- 1. SETTINGS & ASSETS ---
# Expanded list with popular symbols from different markets
SYMBOLS = [
    #"BTC-USD", "ETH-USD", "SOL-USD",  # Crypto
    "^NSEI", "RELIANCE.NS", "HDFCBANK.NS", # India (Nifty 50, Reliance, HDFC)
    #"AAPL", "TSLA", "NVDA", "MSFT"      # US (Apple, Tesla, Nvidia, Microsoft)
]

INTERVAL = "5m"
MIN_PROBABILITY = 30 #(default 60%) → trade is skipped and counted as "Skipped" in the dashboard. Raise this to 70–75% to be even more selective.
ATR_SL_MULTIPLIER = 1.5 # try 2.0 if SL still hits too often
TP_THRESHOLD = 0.03  # 3% Fixed Take Profit
IST = ZoneInfo("Asia/Kolkata")

TOKEN = userdata.get("TELEGRAM_TOKEN")
CHAT_ID = userdata.get("CHAT_ID")

logging.basicConfig(level=logging.WARNING)

# --- 2. CORE FUNCTIONS ---
def detect_profile(symbol: str) -> dict:
    s = symbol.upper()
    if "BTC" in s or "ETH" in s or "SOL" in s or s.endswith("-USD"):
        return dict(type="CRYPTO", tz="UTC", hours=None, doji=0.15, strike=1)
    if "^NSE" in s or s.endswith(".NS"):
        return dict(type="INDIA", tz="Asia/Kolkata", hours=("09:15","15:30"), doji=0.20, strike=50)
    return dict(type="US_STOCK", tz="America/New_York", hours=("09:30","16:00"), doji=0.20, strike=1)

def is_market_open(profile: dict) -> bool:
    if profile["hours"] is None: return True
    now_tz = datetime.now(ZoneInfo(profile["tz"]))
    if now_tz.weekday() >= 5: return False
    curr = now_tz.strftime("%H:%M")
    return profile["hours"][0] <= curr <= profile["hours"][1]

def send_alert(msg):
    if not TOKEN or not CHAT_ID: print(f"[LOG] {msg}"); return
    try: requests.post(f"https://api.telegram.org/bot{TOKEN}/sendMessage", data={"chat_id": CHAT_ID, "text": msg}, timeout=5)
    except: pass

def fetch_data(symbol, interval):
    try:
        df = yf.download(symbol, period="2d", interval=interval, auto_adjust=False, progress=False)
        if isinstance(df.columns, pd.MultiIndex): df.columns = [c[0] for c in df.columns]
        return df if not df.empty else None
    except: return None

def heikin_ashi(df):
    # Select relevant columns and drop NaNs. This creates one copy.
    # Assuming yfinance already returns numeric columns, .apply(pd.to_numeric) is often redundant.
    ohlc_df = df[["Open", "High", "Low", "Close"]].dropna()

    if len(ohlc_df) < 10:
        return None

    # Extract NumPy arrays directly from the cleaned DataFrame to avoid further Pandas overhead
    open_prices = ohlc_df["Open"].values
    high_prices = ohlc_df["High"].values
    low_prices = ohlc_df["Low"].values
    close_prices = ohlc_df["Close"].values

    ha_close = (open_prices + high_prices + low_prices + close_prices) / 4

    ha_open = np.zeros_like(open_prices)
    ha_open[0] = (open_prices[0] + close_prices[0]) / 2
    for i in range(1, len(open_prices)):
        ha_open[i] = (ha_open[i-1] + ha_close[i-1]) / 2

    ha_high = np.maximum.reduce([ha_open, ha_close, high_prices])
    ha_low = np.minimum.reduce([ha_open, ha_close, low_prices])

    return pd.DataFrame({"open": ha_open, "close": ha_close, "high": ha_high, "low": ha_low})

def compute_indicators(df):
    close, high, low = df["Close"], df["High"], df["Low"]
    delta = close.diff(); gain = delta.clip(lower=0).rolling(14).mean(); loss = (-delta.clip(upper=0)).rolling(14).mean()
    rsi = float((100 - 100 / (1 + gain / loss)).iloc[-2])
    tr = pd.concat([(high - low), (high - close.shift(1)).abs(), (low - close.shift(1)).abs()], axis=1).max(axis=1)
    return {"rsi": rsi, "atr": float(tr.rolling(14).mean().iloc[-2]), "ema": float(close.ewm(span=20).mean().iloc[-2])}

def print_dashboard(symbol_states):
    clear_output(wait=True)
    print(f"\n  === MULTI-SYMBOL BOT ({datetime.now(IST).strftime('%I:%M:%S %p')}) ===")
    print(f"  {'SYMBOL':<12} {'STATUS':<8} {'POS':<6} {'ENTRY':<10} {'PRICE':<10} {'UNREAL':<10} {'TRADES':<7} {'P&L':<8} {'WINS':<5} {'LOSSES':<7} {'WIN%':<6} {'BEST':<8} {'WORST':<8}")
    print("  " + "-"*150)

    g_trades, g_pnl = 0, 0.0
    g_wins, g_losses = 0, 0
    g_best_trade = float('-inf')
    g_worst_trade = float('inf')

    for s, st in symbol_states.items():
        m_stat = "OPEN" if is_market_open(st['profile']) else "CLOSED"
        pos, ent, cur = st['position'] or "-", round(st['entry_price'] or 0, 2), round(st['latest_price'] or 0, 2)
        unreal = 0.0
        if pos == "CALL": unreal = cur - ent
        elif pos == "PUT": unreal = ent - cur

        # Symbol-specific metrics
        wins = st['wins']
        losses = st['losses']
        symbol_pnl = st['stats']['pnl']
        best_trade = round(st['best_trade'], 2) if st['best_trade'] != float('-inf') else "N/A"
        worst_trade = round(st['worst_trade'], 2) if st['worst_trade'] != float('inf') else "N/A"

        total_trades = wins + losses
        win_rate = round((wins / total_trades) * 100, 1) if total_trades > 0 else 0.0

        print(f"  {s:<12} {m_stat:<8} {pos:<6} {ent:<10} {cur:<10} {round(unreal,2):<10} {total_trades:<7} {round(symbol_pnl,2):<8} {wins:<5} {losses:<7} {win_rate:<6} {best_trade:<8} {worst_trade:<8}")

        # Update global metrics
        g_trades += total_trades # Sum of total trades for each symbol
        g_pnl += symbol_pnl
        g_wins += wins
        g_losses += losses
        if st['best_trade'] != float('-inf'): g_best_trade = max(g_best_trade, st['best_trade'])
        if st['worst_trade'] != float('inf'): g_worst_trade = min(g_worst_trade, st['worst_trade'])

    # Global Summary
    global_total_trades = g_wins + g_losses
    global_win_rate = round((g_wins / global_total_trades) * 100, 1) if global_total_trades > 0 else 0.0
    g_best_trade_display = round(g_best_trade, 2) if g_best_trade != float('-inf') else "N/A"
    g_worst_trade_display = round(g_worst_trade, 2) if g_worst_trade != float('inf') else "N/A"

    print(f"\n  Global Trades: {g_trades} | Realized P&L: {round(g_pnl, 2)} pts | Wins: {g_wins} | Losses: {g_losses} | Win Rate: {global_win_rate}% | Best Trade: {g_best_trade_display} | Worst Trade: {g_worst_trade_display}")

# --- 3. MAIN LOOP ---
def start_bot():
    states = {s: {
        "position": None,
        "entry_price": None,
        "trailing_sl": None,
        "latest_price": 0,
        "profile": detect_profile(s),
        "stats": {"trades":0, "pnl":0.0},
        "wins": 0,
        "losses": 0,
        "best_trade": float('-inf'),
        "worst_trade": float('inf'),
        "last_time": None
    } for s in SYMBOLS}

    while True:
        for s in SYMBOLS:
            st = states[s]
            if not is_market_open(st["profile"]): continue

            df = fetch_data(s, INTERVAL)
            if df is None: continue
            if df.index[-1] == st["last_time"]: continue
            st["last_time"] = df.index[-1]

            ha = heikin_ashi(df); ind = compute_indicators(df)
            price = float(df["Close"].iloc[-1]); st["latest_price"] = price

            # Exit Logic (TP & SL)
            if st["position"]:
                p_pct = (price - st["entry_price"])/st["entry_price"] if st["position"] == "CALL" else (st["entry_price"] - price)/st["entry_price"]
                if p_pct >= TP_THRESHOLD or (st["position"] == "CALL" and price < st["trailing_sl"]) or (st["position"] == "PUT" and price > st["trailing_sl"]):
                    pnl = (price - st["entry_price"]) if st["position"] == "CALL" else (st["entry_price"] - price)
                    st["stats"]["pnl"] += pnl

                    # Update performance metrics
                    if pnl > 0:
                        st['wins'] += 1
                    else:
                        st['losses'] += 1
                    st['best_trade'] = max(st['best_trade'], pnl)
                    st['worst_trade'] = min(st['worst_trade'], pnl)

                    send_alert(f"EXIT {s} | P&L: {round(pnl,2)}")
                    st["position"] = None; st["entry_price"] = None

            # Entry Logic (Simplified Signal)
            elif ha.iloc[-2]["close"] > ha.iloc[-2]["open"] and ha.iloc[-3]["close"] < ha.iloc[-3]["open"]:
                st["position"], st["entry_price"] = "CALL", price
                st["trailing_sl"] = price - (ind["atr"] * ATR_SL_MULTIPLIER); st["stats"]["trades"] += 1
                send_alert(f"ENTRY CALL {s} at {price}")

        print_dashboard(states)
        time.sleep(30)

start_bot()


  === MULTI-SYMBOL BOT (02:47:33 PM) ===
  SYMBOL       STATUS   POS    ENTRY      PRICE      UNREAL     TRADES  P&L      WINS  LOSSES  WIN%   BEST     WORST   
  ------------------------------------------------------------------------------------------------------------------------------------------------------
  ^NSEI        OPEN     CALL   23909.2    23941.05   31.85      0       0.0      0     0       0.0    N/A      N/A     
  RELIANCE.NS  OPEN     -      0          1416.9     0.0        0       0.0      0     0       0.0    N/A      N/A     
  HDFCBANK.NS  OPEN     CALL   835.5      836.65     1.15       0       0.0      0     0       0.0    N/A      N/A     

  Global Trades: 0 | Realized P&L: 0.0 pts | Wins: 0 | Losses: 0 | Win Rate: 0.0% | Best Trade: N/A | Worst Trade: N/A


### Select Markets to Analyze

Run the following cell to select the markets you want to analyze. Enter numbers separated by commas for your choices. If you don't enter anything, the bot will default to the India market symbols (`^NSEI`, `RELIANCE.NS`, `HDFCBANK.NS`).

In [15]:
market_options = {
    1: {"name": "Crypto", "symbols": ["BTC-USD", "ETH-USD", "SOL-USD"]},
    2: {"name": "India", "symbols": ["^NSEI", "RELIANCE.NS", "HDFCBANK.NS"]},
    3: {"name": "US Stocks", "symbols": ["AAPL", "TSLA", "NVDA", "MSFT"]}
}

print("Available Markets:")
for key, value in market_options.items():
    print(f"{key}. {value['name']}")

user_input = input("Enter market numbers (e.g., 1,2) or press Enter for default (India): ")

selected_symbols = []
if user_input.strip() == "":
    # Default to India market symbols if no input
    selected_symbols = market_options[2]["symbols"]
    print("No selection made. Defaulting to India market symbols.")
else:
    try:
        choices = [int(c.strip()) for c in user_input.split(',')]
        for choice in choices:
            if choice in market_options:
                selected_symbols.extend(market_options[choice]["symbols"])
            else:
                print(f"Invalid choice: {choice}. Skipping.")
        if not selected_symbols:
            # Fallback if all choices were invalid
            selected_symbols = market_options[2]["symbols"]
            print("No valid selection made. Defaulting to India market symbols.")
    except ValueError:
        selected_symbols = market_options[2]["symbols"]
        print("Invalid input format. Defaulting to India market symbols.")

SYMBOLS = list(set(selected_symbols)) # Use set to avoid duplicates, then convert back to list
print(f"Analyzing symbols: {SYMBOLS}")


Available Markets:
1. Crypto
2. India
3. US Stocks
Enter market numbers (e.g., 1,2) or press Enter for default (India): India
Invalid input format. Defaulting to India market symbols.
Analyzing symbols: ['^NSEI', 'RELIANCE.NS', 'HDFCBANK.NS']
